# LangGraph v2 Test: OCR + Vision Pipeline

End-to-end test of the `/vision-service/v1/documents/process` endpoint using the `ocr_vision_pipeline` flow (Document AI first, then Gemini with enriched prompt).

The endpoint accepts **multipart/form-data** with form fields and exactly one input source:
- `file` (local upload), or
- `gcs_uri` (`gs://bucket/path/file`)

In [13]:
# 1. Config: service URL

BASE_URL = "http://127.0.0.1:8080"   # your running microservice
REQUEST_TIMEOUT = 600

# Option A: Local test document upload
LOCAL_PATH = r"C:\Users\user\Documents\Vision Model\Payments Vision Model\Test Payments Doc - EN 2.jpeg"  # adjust path as needed

# Option B: GCS URI input
GCS_URI = "gs://vision_agent/court-order-dummy.pdf"  # adjust path as needed


In [14]:
# 2. Just requests — no helpers (what a consumer would write)

import json
import requests

In [15]:
# 3. GET /health

resp = requests.get(f"{BASE_URL}/vision-service/v1/health", timeout=30)
print(resp.status_code, resp.json())

200 {'status': 'ok', 'version': '1.0.0'}


---
## 4. Test: OCR + Vision Pipeline (multipart/form-data)

In [16]:
# Keep extraction schema empty for now.
# Add your own project-specific schema later when needed.
EXTRACTION_SCHEMA = None


In [17]:
# 4. POST /vision-service/v1/documents/process — multipart/form-data

import json
import requests
import os

data = {
    "flow_type": "ocr_pipeline",
    "processor_type": "form_parser",
    "custom_prompt": None,
    "include_word_confidence": "false",
    "extraction_schema": None,
    "include_image_in_vision": "false"
}

# ------------------------------------------------------------------
# OPTION A: Local upload (file)
# Uncomment this block when testing local file input.
# ------------------------------------------------------------------
# with open(LOCAL_PATH, "rb") as file_handle:
#     files = {
#         "file": (os.path.basename(LOCAL_PATH), file_handle),
#     }

#     resp = requests.post(
#         f"{BASE_URL}/vision-service/v1/documents/process",
#         files=files,
#         data=data,
#         timeout=REQUEST_TIMEOUT,
#     )

# ------------------------------------------------------------------
# OPTION B: GCS URI (no file upload)
# Comment OPTION A above, then uncomment this block for GCS.
# ------------------------------------------------------------------
data_with_gcs = {
    **data,
    "gcs_uri": GCS_URI,
}

resp = requests.post(
    f"{BASE_URL}/vision-service/v1/documents/process",
    data=data_with_gcs,
    timeout=REQUEST_TIMEOUT,
)

print(resp.status_code)
print(resp.text if resp.status_code >= 400 else json.dumps(resp.json(), indent=2, ensure_ascii=False))

200
{
  "flow": "ocr_pipeline",
  "input": {
    "processor_type": "form_parser",
    "mime_type": "application/pdf",
    "custom_prompt": null
  },
  "result": {
    "ocr": {
      "text": "01/11/202201 28\nمحكمة بداية حقوق عمان\n117\nدائرة التنفيذ\nالتاريخ:\n2022/11/01\nالسادة بنك الور\nالمحترمين\nعلما بأحكام المادة 31 من قانون ،التنفيذ تقرر الحجز التنفيذي على أموال المحكوم عليهم بحدود المبالغ المطلوبة\nرقم القضية\nاسم المحكوم عليه\nالرقم الوطني\nالمبلغ كتابة\nملاحظات\nمقدار\nالمبلغ\n9155/2020\nرند حسن سعيد العشي\n2100 9992246589\nالفان ومائة دينار\nاستثناء حساب\nالرواتب\n3500/2021\nقصي سعيد محمود عبده\n2340 9977556886\nألفان وثلاثمائة وأربعون\nدينار\nاستثناء حساب\nالرواتب\n4821/2021\nسامر أحمد يوسف الحسن\n7100 9912345678\nسبعة آلاف ومائة دينار\n7634/2019\nرامي خالد عبد الله منصور\n1575 9934567892\nألف وخمسمائة وخمسة\nاستثناء حساب\nالرواتب\nوسبعون ديناراً\n3342/2018\nعادل محمود حسن الشريف\n4800 9956789124\nأربعة آلاف وثمانمائة دينار\n2415/2020\nإيهاب علي عبد الرحمن سليم 9978912346 92